## Reading Bronze.Hotel_bookings Delta format



In [0]:
hotel_bookings_bronze_path = "s3://travel-analytics-bronze/delta/bronze/hotel_bookings/"
hotel_bookings_bronze_df = spark.read.format("delta").load(hotel_bookings_bronze_path)

## Silver Transformations


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
col, trim, upper, to_date, to_timestamp,
    when, date_format, concat, lit, coalesce, expr
)
# =============================================================
# STEP 0:CONFIGURATION & SETUP
# =============================================================
table_name = "hotel_bookings"
hotel_bookings_silver_path = f"s3://travel-analytics-bronze/delta/silver/{table_name}/"


# =============================================================
# STEP 1: DATA TYPE CASTING & Parsing
# =============================================================
print("\n STEP 1: Casting Data Types...")
hotel_bookings_step_1_df = ( 
    hotel_bookings_bronze_df 

    # ========== Numeric columns ==========
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("hotel_id", col("hotel_id").cast("int"))
    .withColumn("price", col("price").cast("double"))

    # ========== Boolean columns ==========
    .withColumn("breakfast_included", col("breakfast_included").cast("boolean"))

    # ========== String columns ==========
    .withColumn("payment_method", trim(upper(col("payment_method"))))
    .withColumn("payment_status", trim(upper(col("payment_status"))))
    .withColumn("booking_time", col("booking_time").cast("string"))

    # ========== Extracting and Parsing Dates from Structs ==========
    .withColumn("booking_date", to_date(col("booking_date.member0"))) 
    .withColumn("check_in_date", to_date(col("check_in_date.member0"))) 
    .withColumn("check_out_date", to_date(col("check_out_date.member0")))
)

# ===================================================================================
# STEP 2: Cleaning , Standardizing String Columns and  Business Logic applications
# ===================================================================================
print("\n STEP 2: Cleaning , Standardizing String Columns and  Business Logic applications...")

hotel_bookings_step_2_df = (
    hotel_bookings_step_1_df
    
    # Trim and uppercase
    .withColumn("payment_method", trim(upper(coalesce(col("payment_method"), lit("UNKNOWN")))))
    .withColumn("payment_status", trim(upper(coalesce(col("payment_status"), lit("UNKNOWN"))))) 

    # Renaming 
    .withColumnRenamed("_ab_cdc_updated_at", "updated_at")
    # Business Logic
    .withColumn("stay_duration_days", F.datediff(col("check_out_date"), col("check_in_date")))
    .withColumn("booking_lead_time_days",F.datediff(col("check_in_date"), col("booking_date")))
    # Price Per Night (safe division)
    .withColumn("price_per_night",
        when(col("stay_duration_days") > 0,col("price") / col("stay_duration_days") ).otherwise(None)
    )
)
# ===================================================================================
#  Step 3: Deduplicating Using Composite PK("customer_id", "hotel_id", "check_in_date")
#          Dropping unwanted Airbyte metadata columns
# ===================================================================================
print("\n STEP 2: Deduplicating and Dropping unwanted Airbyte metadata columns...")

airbyte_columns_to_drop = [
    "_airbyte_ab_id",
    "_airbyte_emitted_at",
    "_ab_cdc_lsn",
    "_airbyte_additional_properties",
    "_ab_cdc_deleted_at"
]

hotel_bookings_step_3_df = (
    hotel_bookings_step_2_df
     
    # Drop Duplicates and Airbyte Metadata columns`
    .dropDuplicates(["customer_id", "hotel_id", "check_in_date"])
    .drop(*airbyte_columns_to_drop)
)




 STEP 1: Casting Data Types...

 STEP 2: Cleaning , Standardizing String Columns and  Business Logic applications...

 STEP 2: Deduplicating and Dropping unwanted Airbyte metadata columns...


In [0]:
# =============================================================
# STEP 4: COLUMN RENAMING (Business-Friendly Names)
# =============================================================
print("\nSTEP : Renaming Columns to Business Standards...")
rename_map = {
    "customer_id": "Customer_Id",
    "hotel_id": "Hotel_Id",
    "booking_date": "Booking_Date",
    "booking_time": "Booking_Time",
    "check_in_date": "Check_In_Date",
    "check_out_date": "Check_Out_Date",
    "price": "Total_Price",
    "breakfast_included": "Breakfast_Included",
    "payment_method": "Payment_Method",
    "payment_status": "Payment_Status",
    #calculated columns
    "stay_duration_days": "Stay_Duration_Days",
    "booking_lead_time_days": "Booking_Lead_Time_Days",
    "price_per_night": "Price_Per_Night",
    "updated_at": "Updated_At",
    }
hotel_bookings_silver_df = hotel_bookings_step_3_df.select([col(c).alias(rename_map.get(c, c)) for c in hotel_bookings_step_3_df.columns])


STEP : Renaming Columns to Business Standards...


In [0]:
hotel_bookings_silver_df.display()

Total_Price,Hotel_Id,Customer_Id,Booking_Date,Booking_Time,Check_In_Date,Check_Out_Date,Payment_Method,Payment_Status,Updated_At,Breakfast_Included,Stay_Duration_Days,Booking_Lead_Time_Days,Price_Per_Night
2760.0,128,8910434,2018-06-01,11:00:00,2018-06-15,2018-07-08,BANK TRANSFER,COMPLETED,2025-12-12T01:04:52.921893877Z,false,23,14,120.0
900.0,1347,4708785,2011-02-26,10:40:00,2011-03-05,2011-03-14,BANK TRANSFER,COMPLETED,2025-12-12T01:04:52.921893877Z,true,9,7,100.0
2725.0,1104,9670418,2014-03-08,16:45:00,2014-03-15,2014-04-09,CREDIT CARD,CANCELLED,2025-12-12T01:04:52.921893877Z,true,25,7,109.0
2275.0,208,9946411,2015-10-18,10:50:00,2015-10-26,2015-11-20,DEBIT CARD,COMPLETED,2025-12-12T01:04:52.921893877Z,false,25,8,91.0
480.0,1186,3963321,2017-11-10,12:40:00,2017-11-27,2017-12-05,PAYPAL,COMPLETED,2025-12-12T01:04:52.921893877Z,false,8,17,60.0
1287.0,492,2235746,2018-10-20,12:35:00,2018-11-07,2018-11-20,DEBIT CARD,COMPLETED,2025-12-12T01:04:52.921893877Z,false,13,18,99.0
1827.0,235,3191029,2020-08-27,20:20:00,2020-09-06,2020-10-05,PAYPAL,COMPLETED,2025-12-12T01:04:52.921893877Z,true,29,10,63.0
1140.0,730,7663021,2020-12-09,20:35:00,2020-12-27,2021-01-08,DEBIT CARD,COMPLETED,2025-12-12T01:04:52.921893877Z,true,12,18,95.0
2420.0,1268,8922748,2021-01-01,21:40:00,2021-01-30,2021-02-21,PAYPAL,COMPLETED,2025-12-12T01:04:52.921893877Z,false,22,29,110.0
426.0,1244,2911710,2021-12-10,14:40:00,2022-01-06,2022-01-12,PAYPAL,COMPLETED,2025-12-12T01:04:52.921893877Z,true,6,27,71.0


## Writing Silver Hotel_Bookings to Delta Lake with Check-In Date Partitioning

In [0]:
hotel_bookings_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("Check_In_Date") \
    .save(hotel_bookings_silver_path)